In [1]:
import pandas as pd
import re


In [2]:
TWITTER_PATH = (
    "/Users/janneslampe/Desktop/Coding/Master Thesis/Twitter Parliamentarian Database"
)

In [41]:
member_info = TWITTER_PATH + "/2020_member_info.csv"
df = pd.read_csv(member_info, sep=",", encoding="utf-16", dtype={"uid": str}, keep_default_na=False)

/var/folders/yc/yqhl95zn0kg8mmy01bz8fhn00000gn/T/ipykernel_57755/4074623217.py:2: DtypeWarning: Columns (0: is_nationalist) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(member_info, sep=",", encoding="utf-16", dtype={"uid": str}, keep_default_na=False)


In [43]:
df["uid"]

0                  234797704
1                           
2                   15394954
3                   17976923
4                  295685416
                ...         
15352    1109041676042018816
15353              479394745
15354              250740163
15355              435720619
15356               48667895
Name: uid, Length: 15357, dtype: str

In [44]:
import re
import pandas as pd

# 1. Deduplicate column list while preserving order
desired_columns = [
    "name",
    "member_id",
    "party_id",
    "uid",
    "party",
    "name_link",
    "function",
    "region",
    "country",
    "country_id",
    "mp_party_id",
    "id",
    "country_abbr_x",
    "is_current",
    "party_abbr",
    "is_nationalist",
    "political_group",
    "mp_country_x",
    "oecdmember",
    "eumember",
    "edate",
    "date",
    "partyname",
    "partyabbrev",
    "mp_country_party",
    "country_name_short",
    "party_name_short",
    "pg_party_name",
    "party_name_ascii",
    "country_partyabbrev",
]

# Keep only the unique column names
desired_columns = list(dict.fromkeys(desired_columns))

# 2. Handle duplicate column names if pandas added suffixes (e.g., country.1)
# First, remove duplicate columns in the raw merged DataFrame
cleaned_df = df.loc[:, ~df.columns.duplicated()].copy()

# Keep only the requested columns that exist in the DataFrame
available_columns = [
    col for col in desired_columns if col in cleaned_df.columns
]
cleaned_df = cleaned_df[available_columns]


# 3. Clean messy text fields (newlines, excessive whitespace)
def clean_text(val):
    if pd.isna(val):
        return val
    # Replace multiple whitespaces/newlines with a single space
    return re.sub(r"\s+", " ", str(val)).strip()


# Apply cleaning across all string/object columns
object_cols = cleaned_df.select_dtypes(include="object").columns
cleaned_df[object_cols] = cleaned_df[object_cols].map(clean_text)

# 4. Standardize empty strings to NaN
cleaned_df.replace("", pd.NA, inplace=True)

/var/folders/yc/yqhl95zn0kg8mmy01bz8fhn00000gn/T/ipykernel_57755/3239409753.py:61: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = cleaned_df.select_dtypes(include="object").columns


,name,member_id,party_id,uid,party,name_link,function,region,country,country_id,...,edate,date,partyname,partyabbrev,mp_country_party,country_name_short,party_name_short,pg_party_name,party_name_ascii,country_partyabbrev
0,austin scott,101,0,234797704,Republican,NaN,Agriculture Armed Services,Georgia,United States,25,...,02/11/1920,192011,Republican Party,Republicans,United States Republican Party,NaN,NaN,NaN,NaN,United States Republicans
1,glenn w. thompson,103,0,NaN,Republican,NaN,Agriculture Education and the Workforce Natura...,Pennsylvania,United States,25,...,02/11/1920,192011,Republican Party,Republicans,United States Republican Party,NaN,NaN,NaN,NaN,United States Republicans
2,robert e. latta,107,0,15394954,Republican,NaN,Energy and Commerce,Ohio,United States,25,...,02/11/1920,192011,Republican Party,Republicans,United States Republican Party,NaN,NaN,NaN,NaN,United States Republicans
3,cathy mcmorris rodgers,110,0,17976923,Republican,NaN,Energy and Commerce,Washington,United States,25,...,02/11/1920,192011,Republican Party,Republicans,United States Republican Party,NaN,NaN,NaN,NaN,United States Republicans
4,k. michael conaway,114,0,295685416,Republican,NaN,Agriculture Armed Services Intelligence (Perma...,Texas,United States,25,...,02/11/1920,192011,Republican Party,Republicans,United States Republican Party,NaN,NaN,NaN,NaN,United States Republicans
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15352,"Mazón Ramos, José María",16262,602,1109041676042018816,PRC,http://www.congreso.es/portal/page/portal/Cong...,NaN,NaN,Spain,21,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15353,"Oramas González-Moro, Ana María",16291,603,479394745,Cca-NC,http://www.congreso.es/portal/page/portal/Cong...,NaN,NaN,Spain,21,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15354,"Quevedo Iturbe, Pedro",16314,604,250740163,NC-CCa-PNC,http://www.congreso.es/portal/page/portal/Cong...,NaN,NaN,Spain,21,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15355,"Rego Candamil, Néstor",16323,605,435720619,BNG,http://www.congreso.es/portal/page/portal/Cong...,NaN,NaN,Spain,21,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [45]:
eu_members = ['Austria', 'Belgium', 'Denmark', 'Finland', 'France', 'Germany', 'Greece', 'Ireland', 'Italy', 'Latvia', 'Luxembourg', 'Malta', 'Netherlands', 'Poland', 'Slovenia', 'Spain', 'Sweden', 'European Parliament']
eu_members_df = cleaned_df[cleaned_df["country"].isin(eu_members)]
eu_members_df.to_csv(TWITTER_PATH + "/eu_members.csv", index=False, encoding="utf-8")

In [46]:
tweet_ids = TWITTER_PATH + "/all_tweet_ids.csv"
tweet_ids_df = pd.read_csv(tweet_ids, sep=",", encoding="utf-8")

In [47]:
tweet_ids_df.head(10)

,866224163207483392
0,866224414425219072
1,866224432515252225
2,866224455672049664
3,866224459396599809
4,866224838540763137
5,866224965464600576
6,866225018908405760
7,866225040483913728
8,866225475978354690
9,866225496039796736


In [62]:
tweet_ids_2021 = TWITTER_PATH + "/2021.csv"

column_names = [
    "country",
    "party",
    "name",
    "uid",
    "extra",
    "date",
    "tweet_id",
]

df_2021 = pd.read_csv(
    tweet_ids_2021,
    sep=",",
    encoding="utf-8",
    header=None,
    names=column_names,
    dtype={"tweet_id": str, "uid": str},  # Preserves precision of large IDs
    parse_dates=["date"],  # Parses the date column automatically
)

/var/folders/yc/yqhl95zn0kg8mmy01bz8fhn00000gn/T/ipykernel_57755/1636002189.py:13: DtypeWarning: Columns (0: extra) have mixed types. Specify dtype option on import or set low_memory=False.
  df_2021 = pd.read_csv(


In [63]:
df_2021.head(10)
sorted(df_2021["country"].dropna().unique())
df_2021_eu = df_2021[df_2021["country"].isin(eu_members)]
df_2021_eu

,country,party,name,uid,extra,date,tweet_id
1769023,Slovenia,Slovenian Democratic Party Deputy Group,janez janša,258856900,NaN,2021-01-01 12:23:39,1344982617477820417
1769024,Slovenia,Slovenian Democratic Party Deputy Group,janez janša,258856900,NaN,2021-01-01 12:24:04,1344982724432560128
1769025,Slovenia,Slovenian Democratic Party Deputy Group,janez janša,258856900,NaN,2021-01-01 12:24:51,1344982921153814528
1769026,Slovenia,Slovenian Democratic Party Deputy Group,janez janša,258856900,NaN,2021-01-01 12:26:51,1344983421920096256
1769027,Slovenia,Slovenian Democratic Party Deputy Group,janez janša,258856900,NaN,2021-01-01 12:29:08,1344984000176250880
...,...,...,...,...,...,...,...
7555849,Netherlands,VVD,Dilan Yeşilgöz-Zegerius,99354836,\N,2021-08-10 14:59:44,1425109614463438850
7555850,Netherlands,VVD,Dilan Yeşilgöz-Zegerius,99354836,\N,2021-08-10 18:58:03,1425169592712933376
7555851,Netherlands,VVD,Dilan Yeşilgöz-Zegerius,99354836,\N,2021-08-10 19:31:47,1425178079257899013
7555852,Netherlands,VVD,Dilan Yeşilgöz-Zegerius,99354836,\N,2021-08-10 20:19:03,1425189975591620613


In [56]:
tweet_ids_2020 = TWITTER_PATH + "/2020.csv"

column_names = [
    "country",
    "party",
    "name",
    "uid",
    "extra",
    "date",
    "tweet_id",
]

df_2020 = pd.read_csv(
    tweet_ids_2020,
    sep=",",
    encoding="utf-8",
    header=None,
    names=column_names,
    dtype={"tweet_id": str, "uid": str},  # Preserves precision of large IDs
    parse_dates=["date"],  # Parses the date column automatically
)

/var/folders/yc/yqhl95zn0kg8mmy01bz8fhn00000gn/T/ipykernel_57755/2273218635.py:13: DtypeWarning: Columns (0: extra) have mixed types. Specify dtype option on import or set low_memory=False.
  df_2020 = pd.read_csv(


In [65]:
df_2020.head(10)
sorted(df_2020["country"].dropna().unique())
df_2020_eu = df_2020[df_2020["country"].isin(eu_members)]
df_2020_eu

,country,party,name,uid,extra,date,tweet_id
2691540,Slovenia,Slovenian Democratic Party Deputy Group,janez janša,258856900,NaN,2020-01-01 06:41:39,1212262592716128256
2691541,Slovenia,Slovenian Democratic Party Deputy Group,janez janša,258856900,NaN,2020-01-01 06:42:58,1212262923680337920
2691542,Slovenia,Slovenian Democratic Party Deputy Group,janez janša,258856900,NaN,2020-01-01 06:46:24,1212263789246263300
2691543,Slovenia,Slovenian Democratic Party Deputy Group,janez janša,258856900,NaN,2020-01-01 06:46:47,1212263885748756485
2691544,Slovenia,Slovenian Democratic Party Deputy Group,janez janša,258856900,NaN,2020-01-01 06:47:46,1212264132017303554
...,...,...,...,...,...,...,...
11150352,Ireland,Sinn Féin,Aengus Ó Snodaigh,82645739,\N,2020-12-17 11:13:13,1339529075703083009
11150353,Ireland,Sinn Féin,Aengus Ó Snodaigh,82645739,\N,2020-12-18 00:55:51,1339736098159931394
11150354,Ireland,Sinn Féin,Aengus Ó Snodaigh,82645739,\N,2020-12-18 01:00:45,1339737332677828609
11150355,Ireland,Sinn Féin,Aengus Ó Snodaigh,82645739,\N,2020-12-19 11:12:58,1340253789530558466


In [66]:
# 1. Drop the 'extra' column from both DataFrames
df_2020_eu = df_2020_eu.drop(columns=["extra"], errors="ignore")
df_2021_eu = df_2021_eu.drop(columns=["extra"], errors="ignore")

# 2. Concatenate vertically
df_eu_all = pd.concat([df_2020_eu, df_2021_eu], ignore_index=True)

# 3. Export to a combined CSV
df_eu_all.to_csv(
    TWITTER_PATH + "/eu_tweets_2020_2021.csv",
    index=False,
    encoding="utf-8",
)

# Load previsouly created CSV with all tweet ids and speakers

In [6]:
eu_members_path = TWITTER_PATH + "/eu_members.csv"
eu_tweets_path = TWITTER_PATH + "/eu_tweets_2020_2021.csv"

eu_members_df = pd.read_csv(eu_members_path, sep=",", dtype={"uid": str}, encoding="utf-8")
eu_tweets_df = pd.read_csv(eu_tweets_path, sep=",", dtype={"uid": str}, encoding="utf-8")

In [9]:
eu_members_df.sample(5).head(5)

,name,member_id,party_id,uid,party,name_link,function,region,country,country_id,...,edate,date,partyname,partyabbrev,mp_country_party,country_name_short,party_name_short,pg_party_name,party_name_ascii,country_partyabbrev
4701,DAVID COBURN,6104,198,351009963,United Kingdom Independence Party,http://www.europarl.europa.eu/meps/en/124967/D...,NaN,United Kingdom,European Parliament,27,...,07/06/2001,200106.0,United Kingdom Independence Party,UKIP,United Kingdom United Kingdom Independence Party,GBR,UKIP,United Kingdom Independence Party,United Kingdom Independence Party,United Kingdom UKIP
3914,jukka kopra,6000,163,264072635,National Coalition Party,https://www.eduskunta.fi/EN/kansanedustajat/Pa...,NaN,Southeast Finland,Finland,6,...,18/03/1945,194503.0,National Coalition,KK,Finland National Coalition,NaN,NaN,NaN,NaN,Finland KK
8528,Laimer Robert,15959,479,950732132699328256,Social Democratic Party,https://www.parlament.gv.at/WWER/PAD_02330/ind...,NaN,N,Austria,2,...,09/10/1949,194910.0,Austrian Social Democratic Party,SPÖ,Austria Austrian Social Democratic Party,NaN,NaN,NaN,NaN,Austria SPÖ
2326,ezio primo casati,10587,108,1269968502,PARTITO DEMOCRATICO,NaN,NaN,NaN,Italy,12,...,13/04/2008,200804.0,Democratic Party,PD,Italy Democratic Party,ITA,PD,Partito Democratico,Partito Democratico,Italy PD
4676,IOAN MIRCEA PASCU,6133,196,NaN,Partidul Social Democrat,http://www.europarl.europa.eu/meps/en/33984/IO...,NaN,Romania,European Parliament,27,...,20/05/1990,199005.0,Romanian Social Democratic Party,PSDR,Romania Romanian Social Democratic Party,ROU,PSDR,Partidul Social Democrat Român,Partidul Social Democrat Roman,Romania PSDR


In [8]:
eu_tweets_df.sample(5).head(5)

,country,party,name,uid,date,tweet_id
6808783,European Parliament,Platforma Obywatelska,JACEK SARYUSZ-WOLSKI,1485429175,2021-08-18 08:38:50,1427912864862457863
8269155,European Parliament,Christlich Demokratische Union Deutschlands,Sven SCHULZE,2339095172,2021-04-27 18:35:01,1387113070187753473
2280844,Germany,The Left Party,Gregor Gysi,888289790,2020-04-06 07:16:24,1247060571167522817
2883499,Poland,Civic Platform,marek rząsa,1157185489,2020-02-26 13:03:29,1232652403331256322
2810076,Poland,Law and Justice,anna kwiecień,2564000304,2020-12-02 21:14:01,1334244455432544267


In [ ]:
uid_to_find = "888289790"

eu_members_df[eu_members_df["uid"] == uid_to_find]

In [10]:
import requests
from bs4 import BeautifulSoup

def get_tweet_text_oembed(tweet_id):
    url = f"https://publish.twitter.com/oembed?url=https://x.com/i/status/{tweet_id}&omit_script=true"
    response = requests.get(url)

    if response.status_code == 200:
        html = response.json().get("html", "")
        soup = BeautifulSoup(html, "html.parser")
        # Extract main text from the first paragraph tag
        p = soup.find("p")
        return p.get_text() if p else ""
    elif response.status_code == 404:
        return "[Deleted or Private Tweet]"
    return None

get_tweet_text_oembed("1247060571167522817")

'.@GregorGysi: 2011-18 hat EU-Kommission 63x Mitgliedsländer ermahnt, bei Gesundheit zu kürzen/privatisieren. Die im Wortsinn tödlichen Konsequenzen dieses maßgeblich auch von der Bundesregierung beförderten Kürzungskurses tragen im Moment alle in Europa. https://t.co/8nSJU8bWv1 pic.twitter.com/smrZxJnIJG'

In [74]:
import time

for index, row in sample_test.iterrows():
    party = row["party"]
    name = row["name"]
    tweet_id = row["tweet_id"]
    tweet_text = get_tweet_text_oembed(tweet_id)
    print(f"Party: {party}, Name: {name}, Tweet ID: {tweet_id}, Text: {tweet_text}")
    time.sleep(0.5)  # Rate limiting safety pause


Party: CDU/CSU, Name: florian hahn, Tweet ID: 1238084074557882368, Text: Die gute Nachricht des Tages! Der #Verfassungsschutz stuft den #Flügel um #Höcke als das ein was er ist: Rechtsextrem. Oder wie Spitzen der #AfD sagen würden:➡ “Höcke ist die Mitte der Partei" (Gauland)➡ „Flügel ist Bestandteil der Partei" (Chrupalla)@CSU @csu_bt @cducsubt https://t.co/nZ5hMr53NF
Party: Law and Justice, Name: Drabek Przemysław, Tweet ID: 1258782092336549891, Text: 📌 Kiedy 7⃣5⃣ lat temu świat świętował pokonanie Niemiec, w Polsce pełną parą działał zainstalowany przez Sowietów aparat komunistycznego terroru. Radość z pokonania morderców spod znaku swastyki przeradzała się w strach przed czerwonymi wyzwolicielami.#Zniewolenie1945 #8maja pic.twitter.com/3lwYyzV2W0
Party: The Socialist People's Party, Name: kirsten normann andersen, Tweet ID: 1301784807802839041, Text: Vi har haft reelt frit valg siden 2003. Hvordan har det forbedret ældreplejen til det bedre? Mere af det samme 😳 #sundpol
Party: Civic